# Minimal Kaggle Test Notebook for `mclust` via `rpy2`
Use this minimal notebook to test, debug, and verify `mclust` R execution via `rpy2` on Kaggle.

In [ ]:
# 1. Environment Setup
!pip install -q rpy2 pandas numpy scikit-learn

import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri, pandas2ri

# Enable numpy conversion
numpy2ri.activate()

# Pre-install and load R mclust package
print("Installing & loading R 'mclust' package...")
robjects.r('install.packages("mclust", repos="https://cloud.r-project.org", quiet=TRUE)')
robjects.r('suppressPackageStartupMessages(library(mclust))')
print("R package 'mclust' loaded successfully!")

# Define R native helper function
robjects.r('''
run_mclust_native <- function(x, n_clusters, seed=2024) {
    suppressPackageStartupMessages(library(mclust))
    set.seed(seed)
    mat <- as.matrix(x)
    dimnames(mat) <- NULL
    res <- Mclust(mat, G=n_clusters, modelNames="EEE")
    if (is.null(res)) {
        res <- Mclust(mat, G=n_clusters)
    }
    return(as.integer(res$classification))
}
''')
print("Defined R helper function `run_mclust_native`.")


In [ ]:
def test_mclust_python(data_matrix, n_clusters=8, seed=42):
    print(f"\n-------------------------------------------------------")
    print(f"Testing mclust on feature input shape: {data_matrix.shape}")
    print(f"-------------------------------------------------------")
    
    # Apply PCA reduction if feature dimensions > 30
    data_mat = np.array(data_matrix, dtype=np.float64)
    if data_mat.shape[1] > 30:
        n_comps = min(30, data_mat.shape[0] - 1, data_mat.shape[1])
        print(f"Reducing features from {data_mat.shape[1]} to {n_comps} PCA components...")
        pca = PCA(n_components=n_comps, random_state=seed)
        data_mat = pca.fit_transform(data_mat)
        print(f"Reduced matrix shape: {data_mat.shape}")

    # Test Method 1: Native R helper function
    try:
        print("[Method 1] Calling R native function `run_mclust_native`...")
        r_func = robjects.r['run_mclust_native']
        labels = np.array(r_func(data_mat, n_clusters, seed)).astype(str)
        print(f"[SUCCESS] Method 1 returned {len(labels)} labels.")
        print(f"Cluster counts: {pd.Series(labels).value_counts().to_dict()}")
        return labels
    except Exception as e:
        print(f"[FAILED] Method 1 failed: {e}")

    # Test Method 2: Pandas DataFrame transfer
    try:
        print("[Method 2] Converting to pandas DataFrame & running Mclust...")
        df = pd.DataFrame(data_mat)
        pandas2ri.activate()
        r_df = pandas2ri.py2rpy(df)
        robjects.globalenv['test_df'] = r_df
        robjects.r(f'set.seed({seed})')
        robjects.r(f'res <- Mclust(as.matrix(test_df), G={n_clusters}, modelNames="EEE")')
        labels = np.array(robjects.r('res$classification')).astype(str)
        print(f"[SUCCESS] Method 2 returned {len(labels)} labels.")
        print(f"Cluster counts: {pd.Series(labels).value_counts().to_dict()}")
        return labels
    except Exception as e:
        print(f"[FAILED] Method 2 failed: {e}")

    return None

# Execute tests on mock synthetic data matching scGPT shapes (1263 cells x 532 features)
print("=== TEST 1: High-dimensional Feature Matrix (1263 cells x 532 features) ===")
mock_scgpt = np.random.randn(1263, 532)
res1 = test_mclust_python(mock_scgpt, n_clusters=8)

print("\n=== TEST 2: Low-dimensional Matrix (1263 cells x 30 features) ===")
mock_lowdim = np.random.randn(1263, 30)
res2 = test_mclust_python(mock_lowdim, n_clusters=8)
